# OCR Eval DataFrames Starter

Dataframe-first notebook for transcript-backed OCR eval analysis.

Primary local sources:
- `.local/eval_cases/ocr_transcript_cases_review.json`
- `.local/eval_cases/ocr_transcript_cases_all.json`
- `.local/eval_cases/ocr_transcript_cases_growth.json`
- `.local/eval_reports/ocr_transcript_stability.json`
- `.local/eval_reports/ocr_growth_stability.json`
- `.local/runtime_dbs/active/history.db` (`ocr_runs`)

If needed, refresh data first:
`make ocr-data CGPT_EXPORT_ROOT="$HOME/Library/CloudStorage/Dropbox/CGPT-DATA-EXPORT"`


In [ ]:
import json
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display


In [ ]:
def _find_repo_root(start: Path) -> Path:
    probe = start
    for candidate in (probe, *probe.parents):
        if (candidate / '.git').exists():
            return candidate
    return start

def _read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding='utf-8'))

ROOT = _find_repo_root(Path.cwd().resolve())
REVIEW_PATH = ROOT / '.local' / 'eval_cases' / 'ocr_transcript_cases_review.json'
ALL_CASES_PATH = ROOT / '.local' / 'eval_cases' / 'ocr_transcript_cases_all.json'
GROWTH_CASES_PATH = ROOT / '.local' / 'eval_cases' / 'ocr_transcript_cases_growth.json'
STABILITY_LOCKSET_PATH = ROOT / '.local' / 'eval_reports' / 'ocr_transcript_stability.json'
STABILITY_GROWTH_PATH = ROOT / '.local' / 'eval_reports' / 'ocr_growth_stability.json'
HISTORY_DB_PATH = ROOT / '.local' / 'runtime_dbs' / 'active' / 'history.db'

review = _read_json(REVIEW_PATH)
all_cases_raw = _read_json(ALL_CASES_PATH)
growth_cases_raw = _read_json(GROWTH_CASES_PATH)
stability_lockset = _read_json(STABILITY_LOCKSET_PATH)
stability_growth = _read_json(STABILITY_GROWTH_PATH)

def _collect_stability_cases(payload: dict, dataset: str) -> list[dict]:
    cases = payload.get('cases', []) if isinstance(payload, dict) else []
    if not isinstance(cases, list):
        return []
    rows: list[dict] = []
    for case in cases:
        if not isinstance(case, dict):
            continue
        row = dict(case)
        row['dataset'] = dataset
        rows.append(row)
    return rows

episodes = review.get('episodes', []) if isinstance(review, dict) else []
cases_all = all_cases_raw.get('cases', []) if isinstance(all_cases_raw, dict) else []
cases_growth = growth_cases_raw.get('cases', []) if isinstance(growth_cases_raw, dict) else []
stability_cases = (
    _collect_stability_cases(stability_lockset, 'lockset')
    + _collect_stability_cases(stability_growth, 'growth')
)

episodes_df = pd.DataFrame(episodes)
cases_df = pd.DataFrame(stability_cases)
cases_all_df = pd.DataFrame(cases_all)
cases_growth_df = pd.DataFrame(cases_growth)

if HISTORY_DB_PATH.exists():
    with sqlite3.connect(HISTORY_DB_PATH) as conn:
        ocr_runs_df = pd.read_sql_query(
            """
            SELECT
                run_id,
                session_id,
                source_name,
                status,
                length(extracted_text) AS extracted_char_count,
                extracted_text,
                datetime(
                    CASE
                        WHEN created_at > 20000000000 THEN created_at / 1000
                        ELSE created_at
                    END,
                    'unixepoch',
                    'localtime'
                ) AS created_local
            FROM ocr_runs
            ORDER BY created_at DESC
            """,
            conn,
        )
else:
    ocr_runs_df = pd.DataFrame()


In [ ]:
# Dataframe-first high-value views

if cases_df.empty:
    cases_high_value_df = pd.DataFrame()
    cases_summary_df = pd.DataFrame()
else:
    hv_cols = [
        'id', 'dataset', 'lane', 'binary_outcome',
        'pass_rate', 'fail_runs', 'error_runs', 'observed_runs',
        'expected_summary', 'observed_summary', 'variant_preview',
        'source_name', 'image_path',
    ]
    hv_cols = [c for c in hv_cols if c in cases_df.columns]
    cases_high_value_df = cases_df[hv_cols].copy()

    for col in ['fail_runs', 'error_runs', 'observed_runs']:
        if col in cases_high_value_df.columns:
            cases_high_value_df[col] = cases_high_value_df[col].fillna(0).astype(int)
    if 'pass_rate' in cases_high_value_df.columns:
        cases_high_value_df['pass_rate'] = cases_high_value_df['pass_rate'].fillna(0.0).astype(float)

    fail_mask = cases_high_value_df['fail_runs'] > 0 if 'fail_runs' in cases_high_value_df.columns else False
    error_mask = cases_high_value_df['error_runs'] > 0 if 'error_runs' in cases_high_value_df.columns else False
    unstable_mask = cases_high_value_df['pass_rate'] < 1.0 if 'pass_rate' in cases_high_value_df.columns else False
    cases_high_value_df['needs_attention'] = fail_mask | error_mask | unstable_mask

    sort_cols = [c for c in ['needs_attention', 'fail_runs', 'error_runs', 'pass_rate', 'observed_runs', 'id'] if c in cases_high_value_df.columns]
    ascending_map = {
        'needs_attention': False,
        'fail_runs': False,
        'error_runs': False,
        'pass_rate': True,
        'observed_runs': False,
        'id': True,
    }
    cases_high_value_df = cases_high_value_df.sort_values(
        sort_cols,
        ascending=[ascending_map[c] for c in sort_cols],
        kind='stable',
    ).reset_index(drop=True)

    summary_group_cols = [c for c in ['dataset', 'lane'] if c in cases_high_value_df.columns]
    if summary_group_cols:
        cases_summary_df = (
            cases_high_value_df
            .groupby(summary_group_cols, dropna=False)
            .agg(
                total_cases=('id', 'count'),
                attention_cases=('needs_attention', 'sum'),
                avg_pass_rate=('pass_rate', 'mean'),
            )
            .reset_index()
            .sort_values(['attention_cases', 'total_cases'], ascending=[False, False])
        )
        cases_summary_df['avg_pass_rate'] = cases_summary_df['avg_pass_rate'].round(3)
    else:
        cases_summary_df = pd.DataFrame()

if ocr_runs_df.empty:
    ocr_runs_summary_df = pd.DataFrame()
    ocr_fail_runs_df = pd.DataFrame()
else:
    ocr_runs_summary_df = (
        ocr_runs_df.groupby('status', dropna=False)
        .agg(total_runs=("run_id", "count"), avg_chars=("extracted_char_count", "mean"))
        .reset_index()
        .sort_values('total_runs', ascending=False)
    )
    ocr_runs_summary_df['avg_chars'] = ocr_runs_summary_df['avg_chars'].round(1)

    ocr_fail_runs_df = ocr_runs_df[ocr_runs_df['status'].astype(str).str.lower() != 'ok'].copy()
    ocr_fail_runs_df = ocr_fail_runs_df.reset_index(drop=True)


In [ ]:
# Quick preview (optional).
print("episodes_df:", episodes_df.shape)
print("cases_df:", cases_df.shape)
print("cases_all_df:", cases_all_df.shape)
print("cases_growth_df:", cases_growth_df.shape)
print("ocr_runs_df:", ocr_runs_df.shape)

display(cases_summary_df.head(20))
display(cases_high_value_df.head(50))
display(ocr_runs_summary_df.head(20))
display(ocr_fail_runs_df.head(50))
